## Trajectory Analysis

Loads precomputed results from `scripts/analyze_trajectories.py` and produces:
1. PHATE of $z_{enc}$ colored by drift toward escalation centroid
2. Velocity magnitude distribution split by escalation label
3. Prospective probe AUROC bar chart (trajectory vs baseline, per label)
4. Example patient trajectory in PHATE space with velocity arrows

In [ ]:
import json
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path

matplotlib.use("module://matplotlib_inline.backend_inline")
plt.rcParams.update({
    "figure.dpi": 150,
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
})

-- Config --

In [ ]:
MODEL       = "test_01"
EXPERIMENTS = Path("experiments")
MODEL_DIR   = EXPERIMENTS / MODEL
RESULTS_DIR = MODEL_DIR / "results"
FIGURES_DIR = MODEL_DIR / "analysis" / "trajectories" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

-- Load Results --

In [ ]:
with open(RESULTS_DIR / "trajectories.json") as f:
    results = json.load(f)

npz = dict(np.load(RESULTS_DIR / "trajectories.npz", allow_pickle=True))

trajectories   = npz["trajectories"]        # (P, T_max, D)
validity_mask  = npz["validity_mask"]        # (P, T_max)
patient_ids    = npz["patient_ids"]          # (P,)
times          = npz["times"]               # (P, T_max)
vel_mag        = npz["velocity_magnitude"]   # (P, T_max-1)
curvature      = npz["curvature"]            # (P, T_max-2)
arc_lengths    = npz["arc_lengths"]          # (P,)
z_enc_pooled   = npz["z_enc_pooled"]         # (N, D)
subject_ids    = npz["subject_ids"]          # (N,)
mask_pos       = npz["mask_pos"]             # (N,)
label_esc      = npz["label_escalation"]     # (N,)
label_30d      = npz["label_30d_readmit"]    # (N,)

# Drift toward escalation centroid (per trajectory step)
drift_esc_key = "drift_escalation"
drift_esc = npz[drift_esc_key] if drift_esc_key in npz else None

P, T_max, D = trajectories.shape
print(f"Patients: {P}, T_max: {T_max}, D: {D}")
print(f"Samples:  {len(z_enc_pooled)}")
print(f"Labels:   escalation={label_esc.sum()}, 30d={label_30d.sum()}")

### 1. PHATE of z_enc colored by drift toward escalation centroid

Each point is a sample (patient, `mask_pos`). Color = mean drift toward escalation centroid over that patient's trajectory prefix. Top-10 highest-drift patients are highlighted with markers.

In [ ]:
from phate import PHATE

# Compute per-sample mean drift from trajectory data
# Map each sample back to its patient's mean drift value
if drift_esc is not None:
    vel_mask = npz["velocity_mask"]  # (P, T_max-1)
    drift_safe = drift_esc.copy()
    drift_safe[~vel_mask] = np.nan
    patient_mean_drift = np.nanmean(drift_safe, axis=1)  # (P,)

    # Map patient-level drift to sample-level
    pid_to_idx = {str(pid): i for i, pid in enumerate(patient_ids)}
    sample_drift = np.array([
        patient_mean_drift[pid_to_idx[str(sid)]]
        if str(sid) in pid_to_idx else 0.0
        for sid in subject_ids
    ])
else:
    sample_drift = np.zeros(len(z_enc_pooled))

# PHATE embedding
phate_op = PHATE(n_components=2, random_state=42, n_jobs=-1, verbose=0)
z_phate = phate_op.fit_transform(z_enc_pooled)

# Top-10 highest mean drift patients
top10_idx = np.argsort(sample_drift)[-10:]

fig, ax = plt.subplots(figsize=(7, 5.5))
sc = ax.scatter(
    z_phate[:, 0], z_phate[:, 1],
    c=sample_drift, cmap="RdYlBu_r", s=4, alpha=0.6,
    rasterized=True)
ax.scatter(
    z_phate[top10_idx, 0], z_phate[top10_idx, 1],
    edgecolors="black", facecolors="none", s=50, linewidths=1.2,
    label="top-10 drift")
cbar = fig.colorbar(sc, ax=ax, shrink=0.8, pad=0.02)
cbar.set_label("Mean drift toward escalation centroid")
ax.set_xlabel("PHATE 1")
ax.set_ylabel("PHATE 2")
ax.set_title("z_enc colored by drift toward escalation centroid")
ax.legend(loc="upper right", fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "phate_drift_escalation.png", dpi=200)
plt.show()

### 2. Velocity magnitude distribution by escalation label

For each sample, take the velocity at step (mask_pos - 1) in that
patient's trajectory.  Split by whether the masked encounter was an
escalation event.

In [ ]:
# Map samples to their trajectory step velocity
pid_to_idx = {str(pid): i for i, pid in enumerate(patient_ids)}

# For each sample, rebuild its step index in the trajectory
patient_step_counters: dict[str, int] = {}
sample_order = np.argsort(mask_pos)  # sort by mask_pos to rebuild step order

# Build (subject_id, mask_pos) -> trajectory step mapping
patient_steps: dict[str, list[tuple[int, int]]] = {}
for i in range(len(subject_ids)):
    sid = str(subject_ids[i])
    patient_steps.setdefault(sid, []).append((int(mask_pos[i]), i))
for sid in patient_steps:
    patient_steps[sid].sort(key=lambda x: x[0])

# Collect velocity at the step leading into each sample's mask_pos
vel_at_sample = []
esc_at_sample = []
for sid, steps in patient_steps.items():
    if sid not in pid_to_idx:
        continue
    p = pid_to_idx[sid]
    for t, (_, sample_idx) in enumerate(steps):
        if t == 0:
            continue  # no preceding velocity for the first step
        if t - 1 < vel_mag.shape[1] and not np.isnan(vel_mag[p, t - 1]):
            vel_at_sample.append(vel_mag[p, t - 1])
            esc_at_sample.append(int(label_esc[sample_idx]))

vel_at_sample = np.array(vel_at_sample)
esc_at_sample = np.array(esc_at_sample)

fig, ax = plt.subplots(figsize=(6, 4))
bins = np.linspace(0, np.nanpercentile(vel_at_sample, 99), 50)
ax.hist(vel_at_sample[esc_at_sample == 0], bins=bins, alpha=0.6,
        label=f"No escalation (n={int((esc_at_sample == 0).sum())})",
        density=True, color="#4878CF")
ax.hist(vel_at_sample[esc_at_sample == 1], bins=bins, alpha=0.6,
        label=f"Escalation (n={int((esc_at_sample == 1).sum())})",
        density=True, color="#D65F5F")
ax.set_xlabel("Velocity magnitude (L2)")
ax.set_ylabel("Density")
ax.set_title("Velocity magnitude split by escalation label")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "velocity_by_escalation.png", dpi=200)
plt.show()

### 3. Prospective probe AUROC: trajectory features vs baseline

Bar chart comparing trajectory-derived probe AUROC against the
static $z_{enc}$[k-1] baseline, for each label.

In [ ]:
probes = results.get("probes", {})
if probes:
    label_names = list(probes.keys())
    traj_aurocs = [probes[l]["traj_auroc"] for l in label_names]
    base_aurocs = [probes[l]["baseline_auroc"] for l in label_names]

    x = np.arange(len(label_names))
    width = 0.35

    fig, ax = plt.subplots(figsize=(max(5, len(label_names) * 1.5), 4))
    bars1 = ax.bar(x - width / 2, traj_aurocs, width,
                   label="Trajectory features", color="#4878CF")
    bars2 = ax.bar(x + width / 2, base_aurocs, width,
                   label="Baseline (z_enc[k-1])", color="#B0B0B0")

    # Annotate delta above each pair
    for i in range(len(label_names)):
        delta = traj_aurocs[i] - base_aurocs[i]
        y_top = max(traj_aurocs[i], base_aurocs[i])
        sign = "+" if delta >= 0 else ""
        ax.text(x[i], y_top + 0.01, f"{sign}{delta:.3f}",
                ha="center", va="bottom", fontsize=7, fontstyle="italic")

    ax.set_xticks(x)
    ax.set_xticklabels(label_names, rotation=15, ha="right")
    ax.set_ylabel("AUROC")
    ax.set_title("Prospective probe: trajectory vs baseline")
    ax.set_ylim(0.4, min(1.0, max(max(traj_aurocs), max(base_aurocs)) + 0.08))
    ax.legend(fontsize=8)
    ax.axhline(0.5, color="gray", ls="--", lw=0.7, alpha=0.5)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "probe_auroc_comparison.png", dpi=200)
    plt.show()
else:
    print("No probe results found in trajectories.json")

### 4. Example patient trajectory in PHATE space

Pick the patient with the longest valid trajectory and overlay
velocity arrows in PHATE coordinates.

In [ ]:
# Find patient with longest trajectory
valid_counts = validity_mask.sum(axis=1)
example_p = int(np.argmax(valid_counts))
n_steps = int(valid_counts[example_p])
example_sid = str(patient_ids[example_p])
print(f"Example patient: {example_sid} ({n_steps} steps)")

# Get sample indices for this patient
example_sample_idx = [
    i for i in range(len(subject_ids))
    if str(subject_ids[i]) == example_sid
]
# Sort by mask_pos
example_sample_idx.sort(key=lambda i: int(mask_pos[i]))

if len(example_sample_idx) >= 2:
    example_phate = z_phate[example_sample_idx]  # (n_steps, 2)
    example_esc = label_esc[example_sample_idx]

    fig, ax = plt.subplots(figsize=(7, 5.5))

    # Background: all samples, light gray
    ax.scatter(z_phate[:, 0], z_phate[:, 1],
               c="lightgray", s=2, alpha=0.3, rasterized=True)

    # Trajectory path with arrows
    for t in range(len(example_phate) - 1):
        dx = example_phate[t + 1, 0] - example_phate[t, 0]
        dy = example_phate[t + 1, 1] - example_phate[t, 1]
        ax.annotate(
            "", xy=example_phate[t + 1], xytext=example_phate[t],
            arrowprops=dict(arrowstyle="->", color="#D65F5F", lw=1.5))

    # Color steps by escalation label
    colors = ["#4878CF" if e == 0 else "#D65F5F" for e in example_esc]
    ax.scatter(example_phate[:, 0], example_phate[:, 1],
               c=colors, s=40, zorder=5, edgecolors="black", linewidths=0.5)

    # Number each step
    for t in range(len(example_phate)):
        ax.annotate(str(t), example_phate[t],
                    textcoords="offset points", xytext=(5, 5),
                    fontsize=7, color="black")

    ax.set_xlabel("PHATE 1")
    ax.set_ylabel("PHATE 2")
    ax.set_title(f"Patient {example_sid} trajectory ({n_steps} encounters)")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "example_trajectory_phate.png", dpi=200)
    plt.show()
else:
    print(f"Patient {example_sid} has too few samples for trajectory plot")